In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
location = spark.table("workspace.bronze.erp_location")

In [0]:
columns_name_mapping = {
    "CID": "customer_number",
    "CNTRY": "country"
}

for old_name, new_name in columns_name_mapping.items():
    location = location.withColumnRenamed(old_name, new_name)

In [0]:
location = location.withColumn("customer_number", F.regexp_replace(col("customer_number"), "-", ""))

In [0]:
location = location.withColumn("country",
                               F.when(col("country") == "US", "USA")
                                .when(col("country") == "DE", "Germany")
                                .when((col("country") == "") | col("country").isNull(), "n/a")
                                .otherwise(col("country"))
                                )

In [0]:
location.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_location")